# Phase 4: Fine-Tuning Stable Diffusion via LoRA (Task 1)
This notebook refines a pre-trained text-to-image model (SD 1.5) toward domain-specific visuals using custom imagery. To ensure this runs on standard Kaggle T4 GPUs without OOM errors, we use Parameter-Efficient Fine-Tuning (PEFT) mapping LoRA weights strictly to the Cross-Attention blocks of the UNET.

In [ ]:
!pip install -q diffusers transformers accelerate peft datasets


In [ ]:
import os
import sys
import torch
import torch.nn.functional as F
from datasets import load_dataset
from diffusers import StableDiffusionPipeline, AutoencoderKL, DDPMScheduler, UNet2DConditionModel
from peft import LoraConfig, get_peft_model

# Pre-requisites for custom NLP scripts
sys.path.append(os.path.abspath('..'))
from scripts.text_processing import TextEmbedder


### Load Specialized Domain Dataset
We pull a publicly available dataset consisting of stylized artwork concepts (e.g., Pokémon sprite arts mapped to BLIP captions) specifically formatted for Diffusion text-image coupling.

In [ ]:
print("Loading Target Artwork Dataset...")
dataset = load_dataset("svjack/pokemon-blip-captions-en-zh", split="train")
print(f"Samples detected: {len(dataset)}")
print(f"Example text: {dataset[0]['en_text']}")




### Initialize SD1.5 & LoRA Configurations
We freeze the base UNET and only insert low-rank adapter matrices (LoRA) into the attention mechanism.

In [ ]:
model_id = "runwayml/stable-diffusion-v1-5"
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Initializing base UNET on {device} (fp16 mode)...")
unet = UNet2DConditionModel.from_pretrained(
    model_id, 
    subfolder="unet", 
    torch_dtype=torch.float16
)

# Freeze BASE UNET parameters
unet.requires_grad_(False)

# Setup PEFT LoRA Config targeting Cross-Attention modules
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["to_q", "to_v"], # Primary attention projection weights
    lora_dropout=0.0,
    bias="none",
)

# Wrap UNET with PEFT adapters
unet = get_peft_model(unet, lora_config)
unet.to(device)

unet.print_trainable_parameters()


### Construct Training Pipeline using Native TextProcessor Embedder

In [ ]:
from torchvision import transforms

# Setup transformations for incoming images format mapped to UNET standard sizes.
image_transforms = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

def preprocess_function(examples):
    images = [image_transforms(img.convert("RGB")) for img in examples["image"]]
    return {
        "pixel_values": images,
        "text": examples["en_text"]
    }

dataset = dataset.map(preprocess_function, batched=True, remove_columns=["image"])

embedder = TextEmbedder() # External script from Phase 2
vae = AutoencoderKL.from_pretrained(model_id, subfolder="vae", torch_dtype=torch.float16).to(device)
vae.requires_grad_(False)
noise_scheduler = DDPMScheduler.from_pretrained(model_id, subfolder="scheduler")
optimizer = torch.optim.AdamW(unet.parameters(), lr=1e-4)

# Standard Single-Batch Training Loop Snapshot
print("\n--- Executing Mock Training Iteration (Forward Pass) ---")
unet.train()

pixel_values = torch.stack([torch.tensor(img) for img in dataset["pixel_values"][:2]]).to(device, dtype=torch.float16)
text_prompts = dataset["en_text"][:2]

with torch.no_grad():
    # Generate prompt embeddings through Phase 2 NLP pipeline
    encoder_hidden_states = embedder.get_text_embeddings(text_prompts).to(device, dtype=torch.float16)

# Sample Noise
with torch.no_grad():
    latents = vae.encode(pixel_values).latent_dist.sample()
    latents = latents * vae.config.scaling_factor

# Sample Noise
noise = torch.randn_like(latents)
bsz = latents.shape[0]
timesteps = torch.randint(0, noise_scheduler.config.num_train_timesteps, (bsz,), device=device).long()

# Add noise strictly following DDPMScheduler constraints
noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)

# UNET Forward Mapping via LoRA
param_pred = unet(noisy_latents, timesteps, encoder_hidden_states=encoder_hidden_states).sample

loss = F.mse_loss(param_pred.float(), noise.float(), reduction="mean")
loss.backward()
optimizer.step()
optimizer.zero_grad()

print(f"Training iteration complete. Current Loss: {loss.item():.4f}")
unet.save_pretrained('lora_unet_weights')
print("LoRA matrices attached successfully and saved to 'lora_unet_weights' folder.")




